# SAM3 Multi-Camera Courtship Segmentation & 3D Distance

**Workflow:**
1. Label male/female flies (pos/neg clicks) on ONE reference camera for ALL bouts upfront
2. Batch process: reproject labels to all 7 cameras via DLT triangulation, run SAM3 tracking, triangulate 3D COM
3. Review QC per bout

In [ ]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

sam3_root = "/home/eabe/Research/Github/sam3"
if sam3_root not in sys.path:
    sys.path.insert(0, sam3_root)

import os
import tempfile
import shutil

import numpy as np
import pandas as pd
import cv2
import torch
import matplotlib.pyplot as plt
from matplotlib.widgets import RadioButtons
from PIL import Image
from scipy import ndimage
from tqdm.auto import tqdm

from sam3.model_builder import build_sam3_video_model
from sam3.visualization_utils import show_mask, show_points
import utils.io_dict_to_hdf5 as ioh5

In [ ]:
%matplotlib widget

In [ ]:
# ---- Configuration ----
VIDEO_DIR = "/data2/users/eabe/datasets/Johnson_lab/courtship/Predictions_3D_20260123-093310/videos"
BOUT_CSV = "/data2/users/eabe/datasets/Johnson_lab/courtship/Predictions_3D_20260123-093310/courtship_bouts_summary.csv"
CAMERA_NAMES = ["Cam2012630", "Cam2012631", "Cam2012853", "Cam2012855", "Cam2012857", "Cam2012861", "Cam2012862"]
REF_CAM = "Cam2012631"  # Reference camera for labeling
FPS = 800
MALE_OBJ_ID = 1
FEMALE_OBJ_ID = 2

In [ ]:
# ---- DLT calibration utilities ----
def read_dlt_csv(csv_path):
    with open(csv_path) as f:
        return [float(line.strip()) for line in f.readlines()[:11] if line.strip()]

def dlt_to_projection_matrix(coeffs):
    P = np.zeros((3, 4))
    P[0, :] = coeffs[0:4]
    P[1, :] = coeffs[4:8]
    P[2, :3] = coeffs[8:11]
    P[2, 3] = 1.0
    return P

def triangulate_3d(points_2d, cameras):
    """DLT triangulation from multi-view 2D points. points_2d: list of (x,y), cameras: list of 3x4 P."""
    n = len(points_2d)
    A = np.zeros((2 * n, 4))
    for i in range(n):
        x, y = points_2d[i]
        P = cameras[i]
        A[2*i] = x * P[2] - P[0]
        A[2*i + 1] = y * P[2] - P[1]
    _, _, Vt = np.linalg.svd(A)
    X = Vt[-1]
    return X[:3] / X[3]

def project_3d_to_2d(point_3d, P):
    """Project 3D point to 2D using projection matrix P."""
    X_h = np.append(point_3d, 1)
    proj = P @ X_h
    return proj[:2] / proj[2]

# Load calibration
proj_matrices = {}
for cam in CAMERA_NAMES:
    coeffs = read_dlt_csv(os.path.join(VIDEO_DIR, f"{cam}_dlt.csv"))
    proj_matrices[cam] = dlt_to_projection_matrix(coeffs)
print(f"Loaded {len(proj_matrices)} camera calibrations")

In [ ]:
# ---- Load bout metadata ----
bouts_df = pd.read_csv(BOUT_CSV)
display(bouts_df)

output_dir = Path(BOUT_CSV).parent / "sam3_segmentation_3d"
output_dir.mkdir(exist_ok=True)

In [ ]:
# ---- Extract first frame of each bout from reference camera ----
ref_video_path = os.path.join(VIDEO_DIR, f"{REF_CAM}.mp4")
cap = cv2.VideoCapture(ref_video_path)

bout_first_frames = {}
for idx, row in bouts_df.iterrows():
    bid = int(row["bout_idx"])
    sf = int(row["start_frame"])
    cap.set(cv2.CAP_PROP_POS_FRAMES, sf)
    ret, frame = cap.read()
    if ret:
        bout_first_frames[bid] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        print(f"Bout {bid}: extracted frame {sf}")
cap.release()
print(f"\nExtracted first frames for {len(bout_first_frames)} bouts from {REF_CAM}")

In [ ]:
# ---- Label ALL bouts: click through each bout's first frame ----
# For each bout:
#   1. Select Male/Female with radio buttons
#   2. LEFT-click = positive (include), RIGHT-click = negative (exclude)
#   3. Add negative clicks on floor/background to exclude them
#   4. "<< Prev" / "Next >>" to navigate between bouts
#   5. "Clear Bout" to erase all clicks on the current bout

from matplotlib.widgets import Button

all_prompts = {}  # {bout_id: {obj_id: {"points": [...], "labels": [...]}}}
bout_ids = sorted(bout_first_frames.keys())
state = {"bout_index": 0, "current_obj": MALE_OBJ_ID}

fig, (ax_ctrl, ax_img) = plt.subplots(1, 2, figsize=(18, 5), gridspec_kw={"width_ratios": [1.5, 10]})

# Radio buttons for fly selection
radio = RadioButtons(ax_ctrl, ["Male (blue)", "Female (red)"], active=0)
obj_colors = {MALE_OBJ_ID: "blue", FEMALE_OBJ_ID: "red"}

def current_bout_id():
    return bout_ids[state["bout_index"]]

def show_bout(bid):
    ax_img.clear()
    ax_img.imshow(bout_first_frames[bid])
    idx = state["bout_index"]
    ax_img.set_title(f"Bout {bid} ({idx+1}/{len(bout_ids)}) — L-click=pos, R-click=neg")
    # Re-draw any existing prompts for this bout
    if bid in all_prompts:
        for oid, data in all_prompts[bid].items():
            color = obj_colors[oid]
            for pt, lbl in zip(data["points"], data["labels"]):
                marker = "*" if lbl == 1 else "x"
                ax_img.plot(pt[0], pt[1], marker, color=color, markersize=12,
                            markeredgewidth=2, markeredgecolor="white" if lbl == 1 else "black")
    fig.canvas.draw()

def on_radio(label):
    state["current_obj"] = MALE_OBJ_ID if "Male" in label else FEMALE_OBJ_ID

def on_click(event):
    if event.inaxes != ax_img:
        return
    bid = current_bout_id()
    oid = state["current_obj"]
    if bid not in all_prompts:
        all_prompts[bid] = {}
    if oid not in all_prompts[bid]:
        all_prompts[bid][oid] = {"points": [], "labels": []}

    lbl = 1 if event.button == 1 else 0
    all_prompts[bid][oid]["points"].append([event.xdata, event.ydata])
    all_prompts[bid][oid]["labels"].append(lbl)

    color = obj_colors[oid]
    marker = "*" if lbl == 1 else "x"
    ax_img.plot(event.xdata, event.ydata, marker, color=color, markersize=12,
                markeredgewidth=2, markeredgecolor="white" if lbl == 1 else "black")
    fig.canvas.draw()

def go_prev(event):
    if state["bout_index"] > 0:
        state["bout_index"] -= 1
        show_bout(current_bout_id())

def go_next(event):
    if state["bout_index"] < len(bout_ids) - 1:
        state["bout_index"] += 1
        show_bout(current_bout_id())

def clear_bout(event):
    bid = current_bout_id()
    if bid in all_prompts:
        del all_prompts[bid]
    show_bout(bid)

# Navigation buttons
ax_prev = fig.add_axes([0.02, 0.02, 0.06, 0.05])
ax_clear = fig.add_axes([0.09, 0.02, 0.08, 0.05])
ax_next = fig.add_axes([0.18, 0.02, 0.06, 0.05])

btn_prev = Button(ax_prev, "<< Prev")
btn_clear = Button(ax_clear, "Clear Bout")
btn_next = Button(ax_next, "Next >>")

btn_prev.on_clicked(go_prev)
btn_clear.on_clicked(clear_bout)
btn_next.on_clicked(go_next)

radio.on_clicked(on_radio)
cid = fig.canvas.mpl_connect("button_press_event", on_click)
show_bout(current_bout_id())
plt.show()

In [ ]:
# ---- Review all labels ----
try:
    fig.canvas.mpl_disconnect(cid)
except:
    pass

for bid in sorted(all_prompts.keys()):
    print(f"\nBout {bid}:")
    for oid, data in all_prompts[bid].items():
        name = "Male" if oid == MALE_OBJ_ID else "Female"
        pos = sum(data["labels"])
        neg = len(data["labels"]) - pos
        print(f"  {name}: {pos} positive, {neg} negative")

In [ ]:
# ---- Build SAM3 model ----
sam3_model = build_sam3_video_model()
predictor = sam3_model.tracker
predictor.backbone = sam3_model.detector.backbone
print("SAM3 tracker ready")

In [ ]:
# ---- Helper: auto-detect fly blobs for non-reference cameras ----
def detect_blobs(frame_rgb, min_area=50):
    """Find dark blobs via Otsu threshold. Returns list of (center_yx, area, mask)."""
    gray = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2GRAY)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=1)
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=2)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    blobs = []
    for c in contours:
        area = cv2.contourArea(c)
        if area < min_area:
            continue
        mask = np.zeros(gray.shape, dtype=np.uint8)
        cv2.drawContours(mask, [c], -1, 255, -1)
        mask_bool = mask > 0
        com = ndimage.center_of_mass(mask_bool)  # (y, x)
        blobs.append({"center_yx": com, "area": area, "mask": mask_bool})
    blobs.sort(key=lambda b: b["area"], reverse=True)
    return blobs[:5]  # Keep top 5 largest


def match_blobs_to_flies(ref_male_com_xy, ref_female_com_xy, ref_cam, other_cam, blobs):
    """Match detected blobs to male/female using triangulation + reprojection error.
    
    For each pair of blobs, try both assignments (blob_A=male, blob_B=female and vice versa).
    Triangulate each assignment and pick the one with lowest reprojection error.
    """
    P_ref = proj_matrices[ref_cam]
    P_other = proj_matrices[other_cam]

    if len(blobs) < 2:
        return None

    best_score = float("inf")
    best_assignment = None

    # Try all pairs of blobs
    for i in range(len(blobs)):
        for j in range(len(blobs)):
            if i == j:
                continue
            # Assignment: blob i = male, blob j = female
            male_xy = (blobs[i]["center_yx"][1], blobs[i]["center_yx"][0])  # (x, y)
            female_xy = (blobs[j]["center_yx"][1], blobs[j]["center_yx"][0])

            # Triangulate male using ref + other camera
            male_3d = triangulate_3d([ref_male_com_xy, male_xy], [P_ref, P_other])
            female_3d = triangulate_3d([ref_female_com_xy, female_xy], [P_ref, P_other])

            # Reprojection error back to reference camera
            male_reproj = project_3d_to_2d(male_3d, P_ref)
            female_reproj = project_3d_to_2d(female_3d, P_ref)

            err_male = np.linalg.norm(np.array(ref_male_com_xy) - male_reproj)
            err_female = np.linalg.norm(np.array(ref_female_com_xy) - female_reproj)
            total_err = err_male + err_female

            if total_err < best_score:
                best_score = total_err
                best_assignment = {
                    "male_center_yx": blobs[i]["center_yx"],
                    "female_center_yx": blobs[j]["center_yx"],
                    "male_3d": male_3d,
                    "female_3d": female_3d,
                    "reproj_error": total_err,
                }

    return best_assignment

print("Helpers defined")

In [ ]:
# ============================================================
# ---- BATCH PROCESSING: all bouts ----
# ============================================================
all_bout_results = {}

for bout_row_idx, (_, bout) in enumerate(bouts_df.iterrows()):
    bid = int(bout["bout_idx"])
    if bid not in all_prompts:
        print(f"\n--- Bout {bid}: SKIPPED (no labels) ---")
        continue

    start_frame = int(bout["start_frame"])
    nf = int(bout["n_frames"])
    print(f"\n{'='*60}")
    print(f"Bout {bid}: {nf} frames ({bout['duration_s']:.2f}s)")
    print(f"{'='*60}")

    # --- Step 1: Extract frames from all cameras ---
    print("[1/5] Extracting frames from all cameras...")
    cam_frames = {}
    for cam in CAMERA_NAMES:
        cap = cv2.VideoCapture(os.path.join(VIDEO_DIR, f"{cam}.mp4"))
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
        frames = []
        for _ in range(nf):
            ret, frame = cap.read()
            if not ret:
                break
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        cap.release()
        cam_frames[cam] = frames
    print(f"  Extracted from {len(cam_frames)} cameras")

    # --- Step 2: Run SAM3 on reference camera to get mask COM ---
    print(f"[2/5] Running SAM3 on reference camera ({REF_CAM})...")
    ref_frames = cam_frames[REF_CAM]
    h, w = ref_frames[0].shape[:2]

    # Save ref camera frames as JPEGs
    ref_jpeg_dir = tempfile.mkdtemp(prefix=f"sam3_ref_{bid}_")
    for i, frame in enumerate(ref_frames):
        Image.fromarray(frame).save(os.path.join(ref_jpeg_dir, f"{i:06d}.jpg"))

    # Init tracker on ref camera with user prompts
    ref_state = predictor.init_state(video_path=ref_jpeg_dir)
    for oid, data in all_prompts[bid].items():
        pts = np.array(data["points"], dtype=np.float32)
        lbls = np.array(data["labels"], dtype=np.int32)
        rel_pts = [[x / w, y / h] for x, y in pts]
        predictor.add_new_points_or_box(
            inference_state=ref_state, frame_idx=0, obj_id=oid,
            points=torch.tensor(rel_pts, dtype=torch.float32),
            labels=torch.tensor(lbls, dtype=torch.int32),
            clear_old_points=False,
        )

    # Propagate on reference camera
    ref_segments = {}
    for fidx, oids, _, vr_masks, _ in predictor.propagate_in_video(
        ref_state, start_frame_idx=0, max_frame_num_to_track=nf, reverse=False, propagate_preflight=True,
    ):
        ref_segments[fidx] = {
            int(oid): (vr_masks[i] > 0.0).cpu().numpy().squeeze()
            for i, oid in enumerate(oids)
        }

    # Get frame-0 COM from ref camera masks for reprojection
    ref_male_com = ndimage.center_of_mass(ref_segments[0][MALE_OBJ_ID])  # (y, x)
    ref_female_com = ndimage.center_of_mass(ref_segments[0][FEMALE_OBJ_ID])  # (y, x)
    ref_male_xy = (ref_male_com[1], ref_male_com[0])  # (x, y)
    ref_female_xy = (ref_female_com[1], ref_female_com[0])
    print(f"  Ref male COM: ({ref_male_xy[0]:.0f}, {ref_male_xy[1]:.0f})")
    print(f"  Ref female COM: ({ref_female_xy[0]:.0f}, {ref_female_xy[1]:.0f})")
    shutil.rmtree(ref_jpeg_dir)

    # --- Step 3: Match blobs in other cameras via reprojection ---
    print("[3/5] Matching flies across cameras via reprojection...")
    cam_prompts = {REF_CAM: all_prompts[bid]}  # ref camera uses user prompts directly

    for cam in CAMERA_NAMES:
        if cam == REF_CAM:
            continue
        blobs = detect_blobs(cam_frames[cam][0])
        match = match_blobs_to_flies(ref_male_xy, ref_female_xy, REF_CAM, cam, blobs)
        if match is None:
            print(f"  {cam}: FAILED to match (not enough blobs)")
            continue

        mc = match["male_center_yx"]
        fc = match["female_center_yx"]
        cam_prompts[cam] = {
            MALE_OBJ_ID: {"points": [[mc[1], mc[0]]], "labels": [1]},
            FEMALE_OBJ_ID: {"points": [[fc[1], fc[0]]], "labels": [1]},
        }
        print(f"  {cam}: matched (reproj_err={match['reproj_error']:.1f}px)")

    # --- Step 4: Run SAM3 tracker on all cameras ---
    print(f"[4/5] Tracking on {len(cam_prompts)} cameras...")
    cam_com = {}

    for cam in tqdm(cam_prompts.keys(), desc=f"Bout {bid} tracking", leave=False):
        frames = cam_frames[cam]
        h_c, w_c = frames[0].shape[:2]

        jpeg_dir = tempfile.mkdtemp(prefix=f"sam3_{cam}_{bid}_")
        for i, frame in enumerate(frames):
            Image.fromarray(frame).save(os.path.join(jpeg_dir, f"{i:06d}.jpg"))

        inf_state = predictor.init_state(video_path=jpeg_dir)

        for oid, data in cam_prompts[cam].items():
            pts = np.array(data["points"], dtype=np.float32)
            lbls = np.array(data["labels"], dtype=np.int32)
            rel_pts = [[x / w_c, y / h_c] for x, y in pts]
            predictor.add_new_points_or_box(
                inference_state=inf_state, frame_idx=0, obj_id=oid,
                points=torch.tensor(rel_pts, dtype=torch.float32),
                labels=torch.tensor(lbls, dtype=torch.int32),
                clear_old_points=False,
            )

        male_com_arr = np.full((nf, 2), np.nan)
        female_com_arr = np.full((nf, 2), np.nan)
        male_areas = np.zeros(nf)
        female_areas = np.zeros(nf)

        for fidx, oids, _, vr_masks, _ in predictor.propagate_in_video(
            inf_state, start_frame_idx=0, max_frame_num_to_track=nf, reverse=False, propagate_preflight=True,
        ):
            for i, oid in enumerate(oids):
                mask = (vr_masks[i] > 0.0).cpu().numpy().squeeze()
                if mask.any():
                    com = ndimage.center_of_mass(mask)
                    area = mask.sum()
                else:
                    com = (np.nan, np.nan)
                    area = 0
                if oid == MALE_OBJ_ID:
                    male_com_arr[fidx] = com
                    male_areas[fidx] = area
                elif oid == FEMALE_OBJ_ID:
                    female_com_arr[fidx] = com
                    female_areas[fidx] = area

        cam_com[cam] = {"male": male_com_arr, "female": female_com_arr,
                        "male_areas": male_areas, "female_areas": female_areas}
        shutil.rmtree(jpeg_dir)

    # --- Step 5: Triangulate 3D ---
    print("[5/5] Triangulating 3D...")
    male_3d = np.full((nf, 3), np.nan)
    female_3d = np.full((nf, 3), np.nan)
    cam_list = list(cam_com.keys())

    for t in range(nf):
        m_pts, m_cams = [], []
        f_pts, f_cams = [], []
        for cam in cam_list:
            mc = cam_com[cam]["male"][t]
            fc = cam_com[cam]["female"][t]
            if not np.isnan(mc).any():
                m_pts.append([mc[1], mc[0]])  # (x, y)
                m_cams.append(proj_matrices[cam])
            if not np.isnan(fc).any():
                f_pts.append([fc[1], fc[0]])
                f_cams.append(proj_matrices[cam])
        if len(m_pts) >= 2:
            male_3d[t] = triangulate_3d(m_pts, m_cams)
        if len(f_pts) >= 2:
            female_3d[t] = triangulate_3d(f_pts, f_cams)

    distances_3d = np.linalg.norm(male_3d - female_3d, axis=1)
    time_s = np.arange(nf) / FPS

    valid = ~np.isnan(distances_3d)
    print(f"  Valid: {valid.sum()}/{nf} ({valid.mean()*100:.1f}%)")
    print(f"  Mean dist: {np.nanmean(distances_3d):.2f}mm, "
          f"min: {np.nanmin(distances_3d):.2f}mm, max: {np.nanmax(distances_3d):.2f}mm")

    # Store results
    results = {
        "male_com_3d": male_3d, "female_com_3d": female_3d,
        "distances_3d": distances_3d, "time_s": time_s,
        "bout_idx": bid, "start_frame": start_frame, "n_frames": nf, "fps": FPS,
    }
    for cam in cam_com:
        results[f"{cam}_male_com"] = cam_com[cam]["male"]
        results[f"{cam}_female_com"] = cam_com[cam]["female"]
        results[f"{cam}_male_areas"] = cam_com[cam]["male_areas"]
        results[f"{cam}_female_areas"] = cam_com[cam]["female_areas"]

    save_path = output_dir / f"bout_{bid:03d}_segmentation_3d.h5"
    ioh5.save(str(save_path), results)
    all_bout_results[bid] = results
    print(f"  Saved: {save_path}")

    # Free memory
    del cam_frames, cam_com, ref_segments

print(f"\n\n{'='*60}")
print(f"BATCH COMPLETE: {len(all_bout_results)} bouts processed")
print(f"{'='*60}")

In [ ]:
# ---- QC: per-bout 3D distance plots ----
# Reload from disk if needed
for idx, row in bouts_df.iterrows():
    bid = int(row["bout_idx"])
    if bid not in all_bout_results:
        h5_path = output_dir / f"bout_{bid:03d}_segmentation_3d.h5"
        if h5_path.exists():
            all_bout_results[bid] = ioh5.load(str(h5_path))

for bid, res in sorted(all_bout_results.items()):
    t = res["time_s"]
    d = res["distances_3d"]
    m3d = res["male_com_3d"]
    f3d = res["female_com_3d"]

    fig, axes = plt.subplots(2, 1, figsize=(14, 6), gridspec_kw={"height_ratios": [2, 1]})

    axes[0].plot(t, d, "k-", linewidth=0.8)
    axes[0].set_ylabel("3D Distance (mm)")
    axes[0].set_title(f"Bout {bid} — Male-Female 3D Distance")
    axes[0].set_xlim(t[0], t[-1])

    for dim, label in enumerate(["X", "Y", "Z"]):
        axes[1].plot(t, m3d[:, dim], "-", label=f"M {label}", linewidth=0.6, alpha=0.7)
        axes[1].plot(t, f3d[:, dim], "--", label=f"F {label}", linewidth=0.6, alpha=0.7)
    axes[1].set_ylabel("Position (mm)")
    axes[1].set_xlabel("Time (s)")
    axes[1].legend(loc="upper right", fontsize=7, ncol=3)

    plt.tight_layout()
    plt.show()

In [ ]:
# ---- Summary: all bouts ----
n_bouts = len(all_bout_results)
if n_bouts > 0:
    fig, axes = plt.subplots(n_bouts, 1, figsize=(14, 2.5 * n_bouts), sharex=False)
    if n_bouts == 1:
        axes = [axes]

    for ax, (bid, res) in zip(axes, sorted(all_bout_results.items())):
        t = res["time_s"]
        d = res["distances_3d"]
        ax.plot(t, d, "k-", linewidth=0.8)
        ax.set_ylabel("mm")
        ax.set_title(f"Bout {bid} ({len(t)} frames, {t[-1]:.2f}s) — "
                     f"mean={np.nanmean(d):.1f}mm", fontsize=10)
        ax.set_xlim(t[0], t[-1])

    axes[-1].set_xlabel("Time (s)")
    plt.suptitle("Male-Female 3D Distance — All Bouts", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("No results found.")